In [3]:
import pandas as pd

# load both stage outputs
sent = pd.read_csv("../data/processed/fomc_sentiment.csv")
spy  = pd.read_csv("../data/processed/spy_daily.csv")

spy.columns = spy.columns.str.lower()

# CRITICAL: date columns come back as strings from CSV.
# string-vs-string merge "works" but is fragile (whitespace, format drift).
# force both to real datetime so the join key is unambiguous.
sent["date"] = pd.to_datetime(sent["date"]).dt.normalize()
spy["date"]  = pd.to_datetime(spy["date"]).dt.normalize()

# sanity check BEFORE merging: shapes + date ranges + dtypes
print("sentiment:", sent.shape, "|", sent["date"].min().date(), "->", sent["date"].max().date())
print("spy      :", spy.shape,  "|", spy["date"].min().date(),  "->", spy["date"].max().date())
print()
print("sent date dtype:", sent["date"].dtype)
print("spy  date dtype:", spy["date"].dtype)
print()
print("sentiment cols:", list(sent.columns))
print("spy cols      :", list(spy.columns))

sentiment: (126, 7) | 2011-01-26 -> 2026-04-29
spy      : (4121, 4) | 2010-01-04 -> 2026-05-21

sent date dtype: datetime64[us]
spy  date dtype: datetime64[us]

sentiment cols: ['date', 'url', 'clean_text', 'prob_pos', 'prob_neg', 'prob_neu', 'sentiment']
spy cols      : ['date', 'close', 'ret', 'ret_next']


In [4]:
# left join: keep every FOMC row, attach SPY returns for that date.
# left (not inner) on purpose -- if a FOMC date has no SPY match,
# we want to SEE the NaN, not silently drop the row.
df = sent.merge(spy, on="date", how="left")

# --- checklist ---
print("rows after merge:", len(df), "(expected 126)")
print()

# how many FOMC dates failed to find a SPY trading day?
n_missing = df["ret"].isna().sum()
print("rows with NaN ret      :", n_missing)
print("rows with NaN ret_next :", df["ret_next"].isna().sum())
print()

# if any are missing, show exactly which FOMC dates didn't match
if n_missing > 0:
    print("FOMC dates with NO SPY match:")
    print(df.loc[df["ret"].isna(), "date"].dt.date.tolist())
else:
    print("every FOMC date matched a SPY trading day.")

rows after merge: 126 (expected 126)

rows with NaN ret      : 1
rows with NaN ret_next : 1

FOMC dates with NO SPY match:
[datetime.date(2020, 3, 15)]


In [5]:
import pandas as pd

emergency = pd.Timestamp("2020-03-15")
print("day of week:", emergency.day_name())   # expect Sunday

# what SPY trading days bracket this FOMC?
window = spy[(spy["date"] >= "2020-03-11") & (spy["date"] <= "2020-03-18")]
print(window[["date", "close", "ret", "ret_next"]].to_string(index=False))

day of week: Sunday
      date      close       ret  ret_next
2020-03-11 250.728714 -0.048748 -0.095677
2020-03-12 226.739746 -0.095677  0.085486
2020-03-13 246.122910  0.085486 -0.109424
2020-03-16 219.191177 -0.109424  0.053992
2020-03-17 231.025757  0.053992 -0.050633
2020-03-18 219.328247 -0.050633  0.002125


In [6]:
# the single NaN is the 2020-03-15 emergency Sunday FOMC (markets closed).
# drop it explicitly -- not a silent dropna() -- so the reason is on record.
# rationale: our entire pipeline assumes "regular FOMC statement -> that day's
# trading session." an emergency weekend meeting breaks that assumption, and
# its market reaction reflects crisis-signaling, not statement sentiment.
emergency_date = pd.Timestamp("2020-03-15")

before = len(df)
df = df[df["date"] != emergency_date].reset_index(drop=True)
after = len(df)

print(f"dropped {before - after} row (2020-03-15 COVID emergency, markets closed)")
print(f"rows: {before} -> {after}")
print()

# confirm fully clean now
print("NaN ret      :", df["ret"].isna().sum())
print("NaN ret_next :", df["ret_next"].isna().sum())

dropped 1 row (2020-03-15 COVID emergency, markets closed)
rows: 126 -> 125

NaN ret      : 0
NaN ret_next : 0


In [7]:
out_path = "../data/processed/fomc_spy.csv"

# what are we about to save? final shape + columns + a peek
print("final shape:", df.shape)
print("columns    :", list(df.columns))
print()
print(df[["date", "sentiment", "ret", "ret_next"]].head().to_string(index=False))
print()

# save
df.to_csv(out_path, index=False)

# verify by reading it back -- don't trust that the write worked
check = pd.read_csv(out_path)
print(f"saved -> {out_path}")
print("re-read shape:", check.shape, "(should match", df.shape, ")")
print("re-read cols :", list(check.columns))

final shape: (125, 10)
columns    : ['date', 'url', 'clean_text', 'prob_pos', 'prob_neg', 'prob_neu', 'sentiment', 'close', 'ret', 'ret_next']

      date  sentiment       ret  ret_next
2011-01-26   0.038285  0.003871  0.002468
2011-03-15   0.446380 -0.011457 -0.018513
2011-04-27   0.531847  0.006528  0.003244
2011-06-22   0.167267 -0.006026 -0.002875
2011-08-09  -0.087466  0.046499 -0.044178

saved -> ../data/processed/fomc_spy.csv
re-read shape: (125, 10) (should match (125, 10) )
re-read cols : ['date', 'url', 'clean_text', 'prob_pos', 'prob_neg', 'prob_neu', 'sentiment', 'close', 'ret', 'ret_next']
